In [0]:
# Databricks Notebook
# 03_gold_transform

from pyspark.sql.functions import (
    col,
    row_number,
    dense_rank,
    avg,
    max,
    min,
    count,
    round,
    when,
    hour,
    date_format
)

from pyspark.sql.window import Window

# ============================================================
# Configuration
# ============================================================

SILVER_FACILITY_TABLE = "smart_carparking.silver_carpark_occupancy"

GOLD_CURRENT_STATUS_TABLE = "smart_carparking.gold_current_facility_status"
GOLD_HOURLY_TRENDS_TABLE = "smart_carparking.gold_hourly_occupancy_trends"
GOLD_FACILITY_RANKING_TABLE = "smart_carparking.gold_facility_occupancy_ranking"
GOLD_CONGESTION_TIMELINE_TABLE = "smart_carparking.gold_congestion_timeline"

# ============================================================
# Read Silver Facility table
# ============================================================

silver_facility_df = spark.table(SILVER_FACILITY_TABLE)

# ============================================================
# Basic filtering for Gold layer
# Gold should only use valid, analytics-ready records
# ============================================================

valid_silver_df = (
    silver_facility_df
    .filter(col("data_quality_status") == "Valid")
    .filter(col("facility_id").isNotNull())
    .filter(col("message_datetime").isNotNull())
    .filter(col("occupancy_rate").isNotNull())
)

# ============================================================
# GOLD 1 - Current Facility Status
# Latest snapshot per facility
# ============================================================

# Window function:
# For each facility, order records from newest to oldest.
latest_facility_window = (
    Window
    .partitionBy("facility_id")
    .orderBy(col("message_datetime").desc(), col("ingestion_timestamp").desc())
)

latest_facility_df = (
    valid_silver_df
    .withColumn("row_num", row_number().over(latest_facility_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

gold_current_df = (
    latest_facility_df

    # Business rule: parking status based on current occupancy
    .withColumn(
        "parking_status",
        when(col("occupancy_rate") >= 95, "Full")
        .when(col("occupancy_rate") >= 85, "Almost Full")
        .when(col("occupancy_rate") >= 70, "Busy")
        .otherwise("Available")
    )

    # Business rule: pressure level for dashboard colour/category
    .withColumn(
        "pressure_level",
        when(col("occupancy_rate") >= 90, "Critical")
        .when(col("occupancy_rate") >= 75, "High")
        .when(col("occupancy_rate") >= 50, "Medium")
        .otherwise("Low")
    )

    # Business recommendation for AI/dashboard text
    .withColumn(
        "recommendation",
        when(
            col("occupancy_rate") >= 95,
            "This facility is currently full."
        )
        .when(
            col("occupancy_rate") >= 85,
            "Limited parking availability."
        )
        .when(
            col("occupancy_rate") >= 70,
            "Parking demand is increasing."
        )
        .otherwise(
            "Parking availability is currently good."
        )
    )
    .orderBy(col("occupancy_rate").desc())
)

gold_current_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_CURRENT_STATUS_TABLE)

display(gold_current_df)

# ============================================================
# GOLD 2 - Hourly Occupancy Trends
# Historical occupancy analytics by facility and hour
# ============================================================

gold_hourly_trends_df = (
    valid_silver_df

    # Extract hour from timestamp
    .withColumn("hour_of_day", hour(col("message_datetime")))

    # Aggregate historical snapshots by facility and hour
    .groupBy(
        "facility_id",
        "facility_name",
        "suburb",
        "hour_of_day"
    )
    .agg(
        round(avg("occupancy_rate"), 2).alias("avg_occupancy_rate"),
        round(avg("available_spaces"), 2).alias("avg_available_spaces"),
        round(avg("occupied_spaces"), 2).alias("avg_occupied_spaces"),
        count("*").alias("snapshot_count")
    )

    # Categorise historical congestion pressure
    .withColumn(
        "peak_pressure_level",
        when(col("avg_occupancy_rate") >= 90, "Critical")
        .when(col("avg_occupancy_rate") >= 75, "High")
        .when(col("avg_occupancy_rate") >= 50, "Medium")
        .otherwise("Low")
    )
    .orderBy("facility_name", "hour_of_day")
)

gold_hourly_trends_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_HOURLY_TRENDS_TABLE)

display(gold_hourly_trends_df)

# ============================================================
# GOLD 3 - Facility Occupancy Ranking
# Ranks facilities by average historical occupancy
# ============================================================

facility_ranking_df = (
    valid_silver_df
    .groupBy(
        "facility_id",
        "facility_name",
        "suburb"
    )
    .agg(
        round(avg("occupancy_rate"), 2).alias("avg_occupancy_rate"),
        round(max("occupancy_rate"), 2).alias("max_occupancy_rate"),
        round(min("occupancy_rate"), 2).alias("min_occupancy_rate"),
        count("*").alias("snapshot_count")
    )
)

ranking_window = Window.orderBy(col("avg_occupancy_rate").desc())

facility_ranking_df = (
    facility_ranking_df
    .withColumn("occupancy_rank", dense_rank().over(ranking_window))
    .withColumn(
        "demand_category",
        when(col("avg_occupancy_rate") >= 90, "Consistently Critical")
        .when(col("avg_occupancy_rate") >= 75, "High Demand")
        .when(col("avg_occupancy_rate") >= 50, "Moderate Demand")
        .otherwise("Low Demand")
    )
    .orderBy("occupancy_rank")
)

facility_ranking_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_FACILITY_RANKING_TABLE)

display(facility_ranking_df)

# ============================================================
# GOLD 4 - Congestion Timeline
# Designed for heatmap visualisation
# Helps answer: "When should the city intervene?"
# ============================================================

congestion_timeline_df = (
    valid_silver_df

    # Extract day name and hour for timeline analytics
    .withColumn("day_of_week", date_format(col("message_datetime"), "EEEE"))
    .withColumn("hour_of_day", hour(col("message_datetime")))

    .groupBy(
        "suburb",
        "day_of_week",
        "hour_of_day"
    )
    .agg(
        round(avg("occupancy_rate"), 2).alias("avg_occupancy_rate"),
        round(avg("available_spaces"), 2).alias("avg_available_spaces"),
        count("*").alias("snapshot_count")
    )

    # Risk level for dashboard heatmap
    .withColumn(
        "congestion_risk",
        when(col("avg_occupancy_rate") >= 90, "Critical")
        .when(col("avg_occupancy_rate") >= 75, "High")
        .when(col("avg_occupancy_rate") >= 50, "Medium")
        .otherwise("Low")
    )

    # Simple business explanation
    .withColumn(
        "insight",
        when(
            col("avg_occupancy_rate") >= 90,
            "Very high parking pressure. Intervention may be required."
        )
        .when(
            col("avg_occupancy_rate") >= 75,
            "High parking demand. Monitor closely during this period."
        )
        .when(
            col("avg_occupancy_rate") >= 50,
            "Moderate demand. Parking is still available but usage is increasing."
        )
        .otherwise(
            "Low congestion risk. Parking availability is generally good."
        )
    )
    .orderBy("suburb", "day_of_week", "hour_of_day")
)

congestion_timeline_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_CONGESTION_TIMELINE_TABLE)

display(congestion_timeline_df)

facility_id,facility_name,tfnsw_facility_id,tsn,api_time_seconds_since_2000,park_id,suburb,address,latitude,longitude,total_spots,occupied_spaces,loop_count,transient_vehicles,monthly_vehicles,open_gate_count,message_datetime,ingestion_timestamp,source_system,available_spaces,occupancy_rate,data_quality_status,facility_snapshot_key,parking_status,pressure_level,recommendation
488,Seven Hills,214710TPR001,214710,832720848,4,Seven Hills,Terminus Road,-33.77304548,150.9367514,1613,1301,null,1304,0,0,2026-05-21T23:20:48.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,312,80.66,Valid,70c68c045cc7ae73c3300d2bca5242bba4fefde34b50df7d072d4153f315c222,Busy,High,Parking demand is increasing.
19,Campbelltown Farrow Rd (north),256020TPR001,256020,832685055,1,Campbelltown,Farrow Road,-34.062279,150.815283,68,28,384842,null,null,null,2026-05-21T23:24:15.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,40,41.18,Valid,15c7923f1ae842a94bd3d1a8d788600bc7c68dd0d4c24cbaf3e5d692903c0591,Available,Low,Parking availability is currently good.
26,Tallawong P1,2155384TPR001,2155384,832684724,1,Tallawong,Conferta Avenue,-33.69304704,150.9052577,123,28,8929,null,null,null,2026-05-21T23:18:44.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,95,22.76,Valid,1d6daf6dcfba6925da80a930ec17e80956e58a28edac852046f8e8152f718ee3,Available,Low,Parking availability is currently good.
33,Cherrybrook,2126158TPR001,2126158,832684997,1,Cherrybrook,Bradfield Parade,-33.737374,151.033431,384,56,310925,null,null,null,2026-05-21T23:23:17.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,328,14.58,Valid,a63dcf88169785f964b7d31d4342dc636de6c098951f1bf79ea61c1d0a189bc6,Available,Low,Parking availability is currently good.
7,Kiama,253330TPR001,253330,832662066,1,Kiama,Bong Bong Street,-34.673122,150.854546,42,6,null,null,null,null,2026-05-21T17:01:06.000Z,2026-05-21T07:01:41.419Z,TfNSW Car Park API,36,14.29,Valid,3b622431ea526c7c3654ad808bb88ecd58ec93cb8b05924b049ebd5798286af0,Available,Low,Parking availability is currently good.
20,Campbelltown Hurley St,256020TPR002,256020,832684737,1,Campbelltown,Hurley Street,-34.065798,150.812432,113,15,443320,null,null,null,2026-05-21T23:18:57.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,98,13.27,Valid,03527e6750b6ae210b6684443d71fa015bb0277a4365154b4f4af26ef8dd35a0,Available,Low,Parking availability is currently good.
12,Mona Vale,2103108TPR001,2103108,832680808,1,Mona Vale,Golf Avenue,-33.677567,151.306512,68,7,null,null,null,null,2026-05-21T22:13:28.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,61,10.29,Valid,771e73738711fdecd9c433b1a4b5396528664023fbbfeb30bd7bbc0af0a54e46,Available,Low,Parking availability is currently good.
486,Ashfield,213110TPR001,213110,832720791,1,Ashfield,Brown Street,-33.888104,151.126577,228,19,null,38,0,0,2026-05-21T23:19:51.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,209,8.33,Valid,07d52995232305c7fefa32dfd3822971fa2bd2d678fcee6ef8b944950369d290,Available,Low,Parking availability is currently good.
6,Gordon Henry St (north),207210TPR001,207210,832683252,1,Gordon,Henry Street,-33.757065,151.154662,213,15,null,null,null,null,2026-05-21T22:54:12.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,198,7.04,Valid,86b9d33bc7d83543342429527a6919ccae0ef176d8f058d5edc36f12d26a8219,Available,Low,Parking availability is currently good.
15,Sutherland,223210TPR001,223210,832684768,1,Sutherland,East Parade,-34.02955,151.058409,373,23,null,null,null,null,2026-05-21T23:19:28.000Z,2026-05-21T13:24:32.941Z,TfNSW Car Park API,350,6.17,Valid,32c54c08f6626ba382fa132715f4df540860b64f106913f836926f27023323be,Available,Low,Parking availability is currently good.


facility_id,facility_name,suburb,hour_of_day,avg_occupancy_rate,avg_available_spaces,avg_occupied_spaces,snapshot_count,peak_pressure_level
486,Ashfield,Ashfield,0,10.88,192.5,23.5,2,Low
486,Ashfield,Ashfield,2,2.1,214.86,4.57,14,Low
486,Ashfield,Ashfield,3,4.63,206.0,10.0,1,Low
486,Ashfield,Ashfield,4,5.21,204.75,11.25,4,Low
486,Ashfield,Ashfield,5,7.2,204.9,15.9,10,Low
486,Ashfield,Ashfield,6,12.04,190.0,26.0,1,Low
486,Ashfield,Ashfield,7,39.75,130.17,87.83,6,Low
486,Ashfield,Ashfield,8,16.2,181.0,35.0,1,Low
486,Ashfield,Ashfield,9,25.46,161.0,55.0,1,Low
486,Ashfield,Ashfield,10,19.91,173.0,43.0,1,Low


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


facility_id,facility_name,suburb,avg_occupancy_rate,max_occupancy_rate,min_occupancy_rate,snapshot_count,occupancy_rank,demand_category
19,Campbelltown Farrow Rd (north),Campbelltown,70.58,100.0,17.65,138,1,Moderate Demand
26,Tallawong P1,Tallawong,57.54,100.0,0.0,138,2,Moderate Demand
20,Campbelltown Hurley St,Campbelltown,46.34,100.0,0.0,138,3,Low Demand
33,Cherrybrook,Cherrybrook,46.02,100.0,0.0,138,4,Low Demand
14,West Ryde,West Ryde,41.29,100.0,0.0,138,5,Low Demand
11,Narrabeen,Narrabeen,40.8,100.0,0.0,128,6,Low Demand
25,Hornsby,Hornsby,40.36,100.0,0.69,137,7,Low Demand
31,Bella Vista,Bella Vista,39.29,100.0,0.0,138,8,Low Demand
32,Hills Showground,Castle Hill,38.51,100.0,0.0,138,9,Low Demand
12,Mona Vale,Mona Vale,37.32,100.0,0.0,112,10,Low Demand


suburb,day_of_week,hour_of_day,avg_occupancy_rate,avg_available_spaces,snapshot_count,congestion_risk,insight
Ashfield,Friday,2,0.0,216.0,1,Low,Low congestion risk. Parking availability is generally good.
Ashfield,Friday,5,6.02,203.0,1,Low,Low congestion risk. Parking availability is generally good.
Ashfield,Friday,7,87.04,28.0,1,High,High parking demand. Monitor closely during this period.
Ashfield,Friday,15,90.28,21.0,1,Critical,Very high parking pressure. Intervention may be required.
Ashfield,Friday,16,97.69,5.0,1,Critical,Very high parking pressure. Intervention may be required.
Ashfield,Friday,17,84.03,34.5,2,High,High parking demand. Monitor closely during this period.
Ashfield,Friday,18,78.24,47.0,1,High,High parking demand. Monitor closely during this period.
Ashfield,Friday,19,73.61,57.0,1,Medium,Moderate demand. Parking is still available but usage is increasing.
Ashfield,Friday,20,60.96,84.33,3,Medium,Moderate demand. Parking is still available but usage is increasing.
Ashfield,Friday,22,29.47,152.33,3,Low,Low congestion risk. Parking availability is generally good.
